# ERGT Colab Phase 3 Short Proof ERGT

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

It runs the four proof-sized ERGT-v1 conditions with a short `1000 step` budget and compares them against the tracked short proof baseline.

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
DEVICE = "cuda"

BASELINE_RESULTS = "runs/phase0_baseline/short_proof_wikitext2/baseline_results.json"
COMPARISON_DIR = "runs/phase3_geo_attention/short_proof_comparison"

RUN_ERGT = True
RUN_COMPARISON = True
FORCE_RERUN = False

ERGT_CONFIGS = {
    "alpha_zero": "configs/ergt_v1/short_proof_alpha_zero.json",
    "real_d": "configs/ergt_v1/short_proof_real_d.json",
    "random_d": "configs/ergt_v1/short_proof_random_d.json",
    "shuffled_d": "configs/ergt_v1/short_proof_shuffled_d.json",
}

ERGT_RESULTS = {
    "alpha_zero": "runs/phase3_geo_attention/short_proof_alpha_zero/metrics.json",
    "real_d": "runs/phase3_geo_attention/short_proof_real_d/metrics.json",
    "random_d": "runs/phase3_geo_attention/short_proof_random_d/metrics.json",
    "shuffled_d": "runs/phase3_geo_attention/short_proof_shuffled_d/metrics.json",
}

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
)
print("repo:", repo_head.strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"],
    check=True,
)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run four short proof ERGT conditions

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


if RUN_ERGT:
    for condition, config_path in ERGT_CONFIGS.items():
        result_path = Path(PROJECT_DIR) / ERGT_RESULTS[condition]
        if result_path.exists() and not FORCE_RERUN:
            print("Skipping existing condition:", condition, result_path)
            continue

        print("\n=== ERGT short proof condition:", condition, "===")
        run_command(
            [
                sys.executable,
                "experiments/train_ergt_v1.py",
                "--config",
                config_path,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_ERGT is False; no ERGT condition executed.")

for condition, path in ERGT_RESULTS.items():
    if not (Path(PROJECT_DIR) / path).exists():
        raise FileNotFoundError(f"Missing {condition} result: {path}")

## 5. Compare against short proof baseline

In [ ]:
baseline_path = Path(PROJECT_DIR) / BASELINE_RESULTS
if not baseline_path.exists():
    raise FileNotFoundError(
        f"Missing tracked short proof baseline: {BASELINE_RESULTS}. Pull latest main."
    )

if RUN_COMPARISON:
    run_command(
        [
            sys.executable,
            "experiments/compare_phase3.py",
            "--baseline",
            BASELINE_RESULTS,
            "--alpha-zero",
            ERGT_RESULTS["alpha_zero"],
            "--real-d",
            ERGT_RESULTS["real_d"],
            "--random-d",
            ERGT_RESULTS["random_d"],
            "--shuffled-d",
            ERGT_RESULTS["shuffled_d"],
            "--output-dir",
            COMPARISON_DIR,
        ]
    )
else:
    print("RUN_COMPARISON is False; no comparison generated.")

## 6. Print key JSON outputs

In [ ]:
def print_json(path):
    path = Path(PROJECT_DIR) / path
    print("\n===", path.relative_to(PROJECT_DIR), "===")
    if not path.exists():
        print("missing")
        return
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    print(json.dumps(payload, indent=2, sort_keys=True)[:8000])


print_json(BASELINE_RESULTS)
for path in ERGT_RESULTS.values():
    print_json(path)
print_json(str(Path(COMPARISON_DIR) / "comparison_results.json"))
print_json(str(Path(COMPARISON_DIR) / "ablation_report.json"))

## 7. Archive light outputs

In [ ]:
light_root = Path("/content/ergt_short_proof_ergt_light")
if light_root.exists():
    shutil.rmtree(light_root)

include_roots = [
    "runs/phase3_geo_attention/short_proof_alpha_zero",
    "runs/phase3_geo_attention/short_proof_real_d",
    "runs/phase3_geo_attention/short_proof_random_d",
    "runs/phase3_geo_attention/short_proof_shuffled_d",
    COMPARISON_DIR,
]

for include_root in include_roots:
    source_dir = Path(PROJECT_DIR) / include_root
    target_dir = light_root / include_root
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in source_dir.rglob("*"):
        if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
            continue
        relative = path.relative_to(source_dir)
        destination = target_dir / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)
        print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive(
    "/content/ergt_short_proof_ergt_light", "zip", light_root
)
print("Light archive ready:", light_archive)